# 03 - Tools: Legal Retrieval, Search, and Scraper

This notebook demonstrates Member 2's tool layer. Everything here runs offline by using mock data.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.agent.tools.legal_retrieval_tool import LegalRetrievalAdapter
from src.agent.tools.web_search_tool import DuckDuckGoSearchTool, build_practical_search_query
from src.agent.tools.web_scraper_tool import WebScraperTool

## 1. Legal Retrieval Adapter

Member 1 will later provide the real `QueryEngineTool`. Member 2 uses an adapter so the workflow does not depend on Member 1's internal implementation.

In [ ]:
mock_legal_results = [
    {
        "document": "Labor Code",
        "clause": "Article 94",
        "status": "Effective",
        "text": "Employers must pay wages directly, fully, and on time.",
        "url": "mock://labor-code/article-94",
    }
]

legal_tool = LegalRetrievalAdapter(mock_results=mock_legal_results)
legal_basis = legal_tool.retrieve("My company has not paid my salary for 2 months.")
print(json.dumps(legal_basis, indent=2, ensure_ascii=False))

## 2. Search Query Builder

The search query should combine legal issue labels with practical Vietnamese labor-law context.

In [ ]:
legal_issues = ["unpaid salary", "forced resignation"]
query = build_practical_search_query("My company has not paid my salary.", legal_issues)
query

## 3. Mocked Web Search

We inject a fake provider so this notebook does not depend on live internet search.

In [ ]:
def mock_search_provider(query: str, max_results: int):
    return [
        {
            "title": "Unpaid salary practical case",
            "url": "https://example.com/unpaid-salary",
            "snippet": "A worker collected evidence and sent a written request.",
        }
    ]

search_tool = DuckDuckGoSearchTool(provider=mock_search_provider)
search_results = search_tool.search(query)
print(json.dumps(search_results, indent=2, ensure_ascii=False))

## 4. Mocked Web Scraper

The scraper accepts a URL and returns a clean title/content excerpt. Here we mock the HTML fetcher.

In [ ]:
mock_html = """
<html>
  <head><title>Unpaid Salary Case</title><script>ignored()</script></head>
  <body>
    <nav>This navigation should be ignored.</nav>
    <article>The worker collected payslips, messages, and the labor contract.</article>
  </body>
</html>
"""

scraper = WebScraperTool(fetcher=lambda url: mock_html)
scraped = scraper.scrape("https://example.com/unpaid-salary")
print(json.dumps(scraped, indent=2, ensure_ascii=False))

## What You Should Understand Now

The Agent workflow is easier to test because each tool has a clear input/output shape and can be mocked.